# VDJBet – Yellow Fever vaccination analysis

Replicates the `vdjbet_snippet.Rmd` analysis in Python:

1. Load LLWNGPMAV / HLA-A\*02 TRB entries from VDJdb
2. Generate a large OLGA pool and build 1 000 Pgen-matched mock reference sets
3. For each YFV donor × timepoint, count overlapping clonotypes and cells
4. Z-test against the mock distribution; plot time courses and a heatmap

In [ ]:
import math
import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mir.biomarkers.vdjbet import (
    VDJBetOverlapAnalysis,
    build_olga_pool,
    generate_mock_key_sets_from_pool,
)
from mir.common.parser import ClonotypeTableParser, load_vdjdb_latest
from mir.common.repertoire import LocusRepertoire

CACHE_DIR = Path("assets/large")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
YFV_DIR = CACHE_DIR / "yfv19"

## 1  Load VDJdb reference (LLWNGPMAV, HLA-A\*02, TRB)

In [ ]:
vdjdb_rep = load_vdjdb_latest(
    epitope="LLWNGPMAV",
    locus="TRB",
    species="HomoSapiens",
    mhc_a_contains="A*02",
)
print(f"Reference: {vdjdb_rep.clonotype_count} unique clonotypes")
print(f"Example: {vdjdb_rep.clonotypes[0].junction_aa}  {vdjdb_rep.clonotypes[0].v_gene}")

## 2  Build OLGA pool (cached)

50 000 OLGA-generated TRB sequences with Pgen values.  
Built once via `build_olga_pool`; subsequent runs load from disk.
Used by both pgen-only and pgen+V+J mock generation.

In [ ]:
POOL_CACHE = CACHE_DIR / "olga_trb_pool_50k.pkl"

if POOL_CACHE.exists():
    with open(POOL_CACHE, "rb") as fh:
        pool = pickle.load(fh)
    print(f"Loaded pool: {len(pool):,} sequences from cache")
else:
    print("Generating OLGA pool (one-time, ~5 min) ...")
    pool = build_olga_pool("TRB", 50_000, seed=42)
    with open(POOL_CACHE, "wb") as fh:
        pickle.dump(pool, fh)
    print(f"Pool of {len(pool):,} sequences saved to {POOL_CACHE}")

# Quick bin summary
from collections import defaultdict
pool_by_bin: dict[int, int] = defaultdict(int)
for rec in pool:
    if not math.isinf(rec["pgen"]):
        pool_by_bin[int(math.floor(rec["pgen"]))] += 1
print(f"Pgen bins: { dict(sorted(pool_by_bin.items())) }")

## 3  Build mock reference sets (cached)

Two null distributions:
- **pgen-only** — each mock matches the Pgen histogram of the VDJdb reference,
  sampling randomly from V/J genes (controls for TCR sequence generation probability).
- **pgen+V+J** — each mock additionally matches V-gene and J-gene usage within
  each Pgen bin (controls for V/J gene bias between the VDJdb reference and
  the OLGA null model).

Both are built via `generate_mock_key_sets_from_pool` using the pre-computed pool.

In [ ]:
MOCKS_PGEN_CACHE = CACHE_DIR / "mock_pgen_1000.pkl"
MOCKS_PVJ_CACHE  = CACHE_DIR / "mock_pvj_1000.pkl"

if MOCKS_PGEN_CACHE.exists():
    with open(MOCKS_PGEN_CACHE, "rb") as fh:
        mock_pgen = pickle.load(fh)
    print(f"Loaded {len(mock_pgen)} pgen-only mock key sets from cache")
else:
    print("Generating 1 000 pgen-only mock reference sets ...")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        mock_pgen = generate_mock_key_sets_from_pool(vdjdb_rep, pool, 1000, seed=42)
    with open(MOCKS_PGEN_CACHE, "wb") as fh:
        pickle.dump(mock_pgen, fh)
    print(f"Saved to {MOCKS_PGEN_CACHE}")

if MOCKS_PVJ_CACHE.exists():
    with open(MOCKS_PVJ_CACHE, "rb") as fh:
        mock_pvj = pickle.load(fh)
    print(f"Loaded {len(mock_pvj)} pgen+V+J mock key sets from cache")
else:
    # pgen+V+J needs a sparser bin index; the 50k pool may need replacement
    # sampling in fine-grained (V, J, bin) cells — this is expected.
    print("Generating 1 000 pgen+V+J mock reference sets ...")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        mock_pvj = generate_mock_key_sets_from_pool(
            vdjdb_rep, pool, 1000, fix_v_usage=True, fix_j_usage=True, seed=42
        )
    with open(MOCKS_PVJ_CACHE, "wb") as fh:
        pickle.dump(mock_pvj, fh)
    print(f"Saved to {MOCKS_PVJ_CACHE}")

print(f"Example pgen-only mock size: {len(mock_pgen[0])}")
print(f"Example pgen+V+J mock size:  {len(mock_pvj[0])}")

## 4  Download YFV dataset from HuggingFace

In [ ]:
if not YFV_DIR.exists() or not any(YFV_DIR.glob("*.airr.tsv.gz")):
    from huggingface_hub import snapshot_download
    print("Downloading isalgo/airr_yfv19 from HuggingFace ...")
    snapshot_download(
        repo_id="isalgo/airr_yfv19",
        repo_type="dataset",
        local_dir=str(YFV_DIR),
    )
    print(f"Downloaded to {YFV_DIR}")
else:
    print(f"Dataset already cached in {YFV_DIR}")

meta = pd.read_csv(YFV_DIR / "metadata.txt", sep="\t")
print(f"{len(meta)} samples:")
meta

## 5  Load YFV samples

In [ ]:
_parser = ClonotypeTableParser()

samples: list[dict] = []
for _, row in meta.iterrows():
    path = YFV_DIR / row["file_name"]
    df = pd.read_csv(path, sep="\t", compression="infer")
    if "locus" in df.columns:
        df = df[df["locus"].fillna("") == "TRB"]
    # Drop rows with missing or empty junction_aa before parse_inner
    # (str(NaN) would become the literal string 'nan')
    df = df.dropna(subset=["junction_aa"])
    df = df[df["junction_aa"].str.strip().str.len() > 0]
    clones = _parser.parse_inner(df)
    rep = LocusRepertoire(clonotypes=clones, locus="TRB", repertoire_id=row["file_name"])
    samples.append({
        "file_name": row["file_name"],
        "donor":     row["donor"],
        "day":       int(row["day"]),
        "replica":   str(row.get("replica", "1")),
        "repertoire": rep,
    })

print(f"Loaded {len(samples)} TRB samples")
print(f"Example: {samples[0]['donor']} day {samples[0]['day']}  "
      f"{samples[0]['repertoire'].clonotype_count:,} clonotypes")

## 6  Compute overlaps — exact match and 1mm

`VDJBetOverlapAnalysis` wraps the reference repertoire and both null
distributions.  Its `score(sample)` method returns an `OverlapResult`
containing all overlap counts, fractions, and z/p-scores — no boilerplate
needed per sample.

| Metric     | Match type | Null        |
|------------|-----------|-------------|
| `n_exact`  | exact      | pgen-only   |
| `n_exact`  | exact      | pgen+V+J    |
| `n_1mm`    | 1mm        | pgen-only   |
| `n_1mm`    | 1mm        | pgen+V+J    |

The real 1mm reference is pre-expanded by `VDJBetOverlapAnalysis` at
construction time; mock key sets stay compact and are expanded on-the-fly
in `count_overlap`.

In [ ]:
analysis = VDJBetOverlapAnalysis(vdjdb_rep, mock_pgen, mock_pvj)
print(f"Reference exact keys: {len(analysis._ref_exact):,}")
print(f"Reference 1mm   keys: {len(analysis._ref_1mm):,}  "
      f"(~{len(analysis._ref_1mm)//len(analysis._ref_exact)}× expansion)")

rows: list[dict] = []
for si, s in enumerate(samples):
    res = analysis.score(s["repertoire"])
    rows.append({
        "donor":   s["donor"],
        "day":     s["day"],
        "replica": s["replica"],
        # totals
        "n_total":  res.n_total,
        "dc_total": res.dc_total,
        # exact match
        "n_exact":       res.n_exact,
        "dc_exact":      res.dc_exact,
        "frac_n_exact":  res.frac_n_exact,
        "frac_dc_exact": res.frac_dc_exact,
        # 1mm match
        "n_1mm":       res.n_1mm,
        "dc_1mm":      res.dc_1mm,
        "frac_n_1mm":  res.frac_n_1mm,
        "frac_dc_1mm": res.frac_dc_1mm,
        # z/p — pgen-only null
        "z_exact_pgen":    res.z_n_exact_pgen,
        "p_exact_pgen":    res.p_n_exact_pgen,
        "z_1mm_pgen":      res.z_n_1mm_pgen,
        "p_1mm_pgen":      res.p_n_1mm_pgen,
        "z_dc_exact_pgen": res.z_dc_exact_pgen,
        "p_dc_exact_pgen": res.p_dc_exact_pgen,
        # z/p — pgen+V+J null
        "z_exact_pvj": res.z_n_exact_pvj,
        "p_exact_pvj": res.p_n_exact_pvj,
        "z_1mm_pvj":   res.z_n_1mm_pvj,
        "p_1mm_pvj":   res.p_n_1mm_pvj,
        # raw mock lists (needed for box plots)
        "mock_pgen_n_exact":  res.mock_pgen_n_exact,
        "mock_pgen_n_1mm":    res.mock_pgen_n_1mm,
        "mock_pvj_n_exact":   res.mock_pvj_n_exact,
        "mock_pvj_n_1mm":     res.mock_pvj_n_1mm,
        # log2 dc + mocks (for dc box plots)
        "dc_exact_log2":           res.dc_exact_log2,
        "dc_1mm_log2":             res.dc_1mm_log2,
        "mock_pgen_dc_exact_log2": res.mock_pgen_dc_exact_log2,
        "mock_pgen_dc_1mm_log2":   res.mock_pgen_dc_1mm_log2,
    })
    if (si + 1) % 10 == 0:
        print(f"  {si + 1}/{len(samples)} samples processed")

df_res = pd.DataFrame(rows)
df_res[["donor", "day", "replica", "n_total",
        "n_exact", "n_1mm", "dc_exact", "dc_1mm"]].to_string(index=False)

## 7  Significance stars

z-scores and p-values are already in `df_res` (computed inside `OverlapResult`).
Here we only add human-readable star columns for display in plots and tables.

In [ ]:
def _stars(p) -> str:
    if p is None:  return ""
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return ""


for col, pcol in [
    ("stars_exact_pgen",    "p_exact_pgen"),
    ("stars_exact_pvj",     "p_exact_pvj"),
    ("stars_1mm_pgen",      "p_1mm_pgen"),
    ("stars_1mm_pvj",       "p_1mm_pvj"),
    ("stars_dc_exact_pgen", "p_dc_exact_pgen"),
]:
    df_res[col] = df_res[pcol].apply(_stars)

df_res[["donor", "day", "replica",
        "z_exact_pgen", "stars_exact_pgen",
        "z_exact_pvj",  "stars_exact_pvj",
        "z_1mm_pgen",   "stars_1mm_pgen",
        "z_1mm_pvj",    "stars_1mm_pvj"]]

## 8  Time-course plots

Four panels: exact/1mm × pgen-only/pgen+V+J null

In [ ]:
donors   = sorted(df_res["donor"].unique())
replicas = sorted(df_res["replica"].unique())
days_all = sorted(df_res["day"].unique())


def _box_width(days: np.ndarray) -> float:
    if len(days) < 2:
        return 3.0
    gaps = np.diff(np.sort(days))
    return float(gaps[gaps > 0].min()) * 0.4


def _plot_timecourse(real_col: str, mock_col: str, stars_col: str,
                     ylabel: str, title: str) -> None:
    fig, axes = plt.subplots(
        len(replicas), len(donors),
        figsize=(4 * len(donors), 3.5 * len(replicas)),
        squeeze=False,
        sharey=False,
    )
    fig.suptitle(title, fontsize=13, y=1.01)

    for ri, replica in enumerate(replicas):
        for di, donor in enumerate(donors):
            ax = axes[ri][di]
            sub = df_res[
                (df_res["donor"] == donor) & (df_res["replica"] == replica)
            ].sort_values("day")

            if sub.empty:
                ax.set_visible(False)
                continue

            days      = sub["day"].values
            real      = sub[real_col].values
            mock_data = list(sub[mock_col])
            width     = _box_width(days)

            ax.boxplot(
                mock_data,
                positions=days,
                widths=width,
                sym="k.",
                patch_artist=True,
                medianprops=dict(color="#888888", linewidth=1.5),
                boxprops=dict(facecolor="#d0e4f7", alpha=0.8),
                flierprops=dict(markersize=2, markerfacecolor="#888888"),
                manage_ticks=False,
            )

            ax.plot(days, real, "-o", color="#d62728",
                    linewidth=2, markersize=6, zorder=5)

            for day, rv, star in zip(days, real, sub[stars_col]):
                if star:
                    ax.text(day, rv + width * 0.1, star,
                            ha="center", va="bottom",
                            fontsize=11, color="#d62728", fontweight="bold")

            ax.set_title(f"{donor}  R{replica}", fontsize=9)
            ax.set_xlabel("Day post-vaccination", fontsize=8)
            ax.set_ylabel(ylabel, fontsize=8)
            ax.set_xticks(days_all)
            ax.tick_params(labelsize=8)

    plt.tight_layout()
    plt.show()


# Panel 1: exact match, pgen-only null
_plot_timecourse(
    real_col="n_exact",
    mock_col="mock_pgen_n_exact",
    stars_col="stars_exact_pgen",
    ylabel="Matched clonotypes",
    title="LLWNGPMAV – exact match, pgen-only null",
)

# Panel 2: 1mm match, pgen-only null
_plot_timecourse(
    real_col="n_1mm",
    mock_col="mock_pgen_n_1mm",
    stars_col="stars_1mm_pgen",
    ylabel="Matched clonotypes (1mm)",
    title="LLWNGPMAV – 1mm match, pgen-only null",
)

# Panel 3: exact match, pgen+V+J null
_plot_timecourse(
    real_col="n_exact",
    mock_col="mock_pvj_n_exact",
    stars_col="stars_exact_pvj",
    ylabel="Matched clonotypes",
    title="LLWNGPMAV – exact match, pgen+V+J null",
)

# Panel 4: 1mm match, pgen+V+J null
_plot_timecourse(
    real_col="n_1mm",
    mock_col="mock_pvj_n_1mm",
    stars_col="stars_1mm_pvj",
    ylabel="Matched clonotypes (1mm)",
    title="LLWNGPMAV – 1mm match, pgen+V+J null",
)

# Cells (duplicate count) – exact match, pgen-only null (log₂ transformed)
_plot_timecourse(
    real_col="dc_exact_log2",
    mock_col="mock_pgen_dc_exact_log2",
    stars_col="stars_dc_exact_pgen",
    ylabel="log₂(matched cells + 1)",
    title="LLWNGPMAV – exact cells (log₂), pgen-only null",
)

## 9  Heatmap: Cohen's d and significance

In [ ]:
def _heatmap(z_col: str, p_col: str, title: str,
             cmap: str = "RdBu_r", clim: tuple = (-4, 4)) -> None:
    df_res["sample_label"] = df_res["donor"] + "  R" + df_res["replica"]
    pivot_z = df_res.pivot_table(
        index="sample_label", columns="day", values=z_col, aggfunc="first"
    )
    pivot_p = df_res.pivot_table(
        index="sample_label", columns="day", values=p_col, aggfunc="first"
    )

    fig, ax = plt.subplots(figsize=(8, max(4, 0.6 * len(pivot_z))))
    im = ax.imshow(
        pivot_z.values.astype(float),
        aspect="auto", cmap=cmap,
        vmin=clim[0], vmax=clim[1],
        interpolation="nearest",
    )
    plt.colorbar(im, ax=ax, label="z-score", shrink=0.8)

    for ri in range(pivot_z.shape[0]):
        for ci in range(pivot_z.shape[1]):
            p = pivot_p.values[ri, ci]
            z = pivot_z.values[ri, ci]
            if pd.notna(p) and float(p) < 0.05 and pd.notna(z) and float(z) > 0:
                ax.add_patch(plt.Rectangle(
                    (ci - 0.5, ri - 0.5), 1, 1,
                    fill=False, edgecolor="black", linewidth=2,
                ))

    ax.set_xticks(range(len(pivot_z.columns)))
    ax.set_xticklabels([f"Day {d}" for d in pivot_z.columns],
                        rotation=45, ha="right")
    ax.set_yticks(range(len(pivot_z.index)))
    ax.set_yticklabels(pivot_z.index)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


_heatmap("z_exact_pgen", "p_exact_pgen",
         "Clonotype overlap (exact, pgen-only null) – z-score")
_heatmap("z_exact_pvj",  "p_exact_pvj",
         "Clonotype overlap (exact, pgen+V+J null) – z-score")
_heatmap("z_1mm_pgen",   "p_1mm_pgen",
         "Clonotype overlap (1mm, pgen-only null) – z-score")
_heatmap("z_1mm_pvj",    "p_1mm_pvj",
         "Clonotype overlap (1mm, pgen+V+J null) – z-score")
_heatmap("z_dc_exact_pgen", "p_dc_exact_pgen",
         "Cells matched (exact, log₂, pgen-only null) – z-score",
         cmap="Reds", clim=(0, 6))

## 10  Summary table

In [ ]:
summary = (
    df_res[[
        "donor", "day", "replica",
        "n_total", "dc_total",
        "n_exact", "dc_exact",
        "frac_n_exact", "frac_dc_exact",
        "n_1mm", "dc_1mm",
        "frac_n_1mm", "frac_dc_1mm",
        "z_exact_pgen", "p_exact_pgen", "stars_exact_pgen",
        "z_exact_pvj",  "p_exact_pvj",  "stars_exact_pvj",
        "z_1mm_pgen",   "p_1mm_pgen",   "stars_1mm_pgen",
        "z_1mm_pvj",    "p_1mm_pvj",    "stars_1mm_pvj",
        "z_dc_exact_pgen", "p_dc_exact_pgen", "stars_dc_exact_pgen",
    ]]
    .copy()
    .sort_values(["donor", "replica", "day"])
    .reset_index(drop=True)
)

for c in ["frac_n_exact", "frac_dc_exact", "frac_n_1mm", "frac_dc_1mm"]:
    summary[c] = summary[c].map("{:.2e}".format)
for c in ["z_exact_pgen", "z_exact_pvj", "z_1mm_pgen", "z_1mm_pvj", "z_dc_exact_pgen"]:
    summary[c] = summary[c].round(2)
for c in ["p_exact_pgen", "p_exact_pvj", "p_1mm_pgen", "p_1mm_pvj", "p_dc_exact_pgen"]:
    summary[c] = summary[c].map("{:.3f}".format)

summary.rename(columns={
    "n_total":  "clonotypes",
    "dc_total": "cells",
    "n_exact":  "matched_exact",
    "dc_exact": "matched_cells_exact",
    "n_1mm":    "matched_1mm",
    "dc_1mm":   "matched_cells_1mm",
    "frac_n_exact":  "frac_exact",
    "frac_dc_exact": "frac_dc_exact",
    "frac_n_1mm":    "frac_1mm",
    "frac_dc_1mm":   "frac_dc_1mm",
}, inplace=True)

pd.set_option("display.max_rows", 60)
summary